[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/13_simple_search/notebooks/aplicaciones/03_laberintos.ipynb)


# Aplicación 03: Generación y Resolución de Laberintos

Los laberintos son el ejemplo más intuitivo de búsqueda. Un laberinto perfecto es exactamente un árbol — hay exactamente un camino entre cualquier par de celdas. Lo generaremos con backtracking y lo resolveremos con BFS y DFS.

## ¿Qué es un laberinto perfecto?

Formalmente, un laberinto perfecto de $n \times m$ celdas es un grafo no dirigido $G = (V, E)$
donde:
- $V$ = conjunto de celdas libres
- $E$ = conexiones entre celdas adyacentes
- $G$ es un **árbol de expansión** del grafo de cuadrícula — conectado y sin ciclos

Esto implica $|E| = |V| - 1$ y que **existe exactamente un camino** entre cualquier par de celdas.

## Algoritmo de generación: Recursive Backtracking

El generador usa una variante de DFS para "tallar" pasajes:
1. Empezar desde una celda arbitraria
2. Moverse a un vecino no visitado aleatoriamente, abriendo el muro entre ellos
3. Si no hay vecinos no visitados, retroceder (backtrack)
4. Repetir hasta haber visitado todas las celdas

El resultado es siempre un árbol de expansión del grafo de cuadrícula.


In [ ]:
# Instalar dependencias (ejecutar solo en Colab)
# !pip install numpy matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import time

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

COLORS = {
    "primary":   "#2563EB",
    "secondary": "#10B981",
    "accent":    "#F59E0B",
    "danger":    "#EF4444",
    "purple":    "#7C3AED",
}

print("Dependencias cargadas.")


---
## Motor de Búsqueda (auto-contenido)

Copiamos la implementación completa de BFS/DFS/IDDFS para que el notebook sea independiente.


In [ ]:
# ============================================================
# MOTOR DE BÚSQUEDA COMPLETO
# ============================================================

class Frontera:
    def push(self, nodo, padre=None): raise NotImplementedError
    def pop(self): raise NotImplementedError
    def contains(self, nodo): raise NotImplementedError
    def is_empty(self): raise NotImplementedError

class ColaDeFrontera(Frontera):
    def __init__(self):
        self.cola = deque()
        self.miembros = set()
    def push(self, nodo, padre=None):
        self.cola.append(nodo); self.miembros.add(nodo)
    def pop(self):
        nodo = self.cola.popleft(); self.miembros.discard(nodo); return nodo
    def contains(self, nodo): return nodo in self.miembros
    def is_empty(self): return len(self.cola) == 0

class PilaDeFrontera(Frontera):
    def __init__(self):
        self.pila = []
        self.miembros = set()
    def push(self, nodo, padre=None):
        self.pila.append(nodo); self.miembros.add(nodo)
    def pop(self):
        nodo = self.pila.pop(); self.miembros.discard(nodo); return nodo
    def contains(self, nodo): return nodo in self.miembros
    def is_empty(self): return len(self.pila) == 0

def reconstruir_camino(padre, meta):
    camino = []
    nodo = meta
    while nodo is not None:
        camino.append(nodo)
        nodo = padre[nodo]
    camino.reverse()
    return camino

def busqueda_generica(problema, frontera):
    inicio = problema.inicio
    frontera.push(inicio)
    explorado = set()
    padre = {inicio: None}
    nodos_expandidos = 0
    max_frontera = 1
    while not frontera.is_empty():
        nodo = frontera.pop()
        if problema.es_meta(nodo):
            return reconstruir_camino(padre, nodo), nodos_expandidos, max_frontera
        explorado.add(nodo)
        nodos_expandidos += 1
        for vecino in problema.vecinos(nodo):
            if vecino not in explorado and not frontera.contains(vecino):
                padre[vecino] = nodo
                frontera.push(vecino, padre=nodo)
        tam = len(getattr(frontera, 'cola', getattr(frontera, 'pila', [])))
        max_frontera = max(max_frontera, tam)
    return None, nodos_expandidos, max_frontera

def bfs(problema): return busqueda_generica(problema, ColaDeFrontera())
def dfs(problema): return busqueda_generica(problema, PilaDeFrontera())

class Problema:
    inicio = None
    def es_meta(self, nodo): raise NotImplementedError
    def vecinos(self, nodo): raise NotImplementedError

class ProblemaCuadricula(Problema):
    def __init__(self, grid, inicio, meta):
        self.grid = grid
        self.inicio = inicio
        self.meta = meta
        self.filas = len(grid)
        self.cols = len(grid[0])
    def es_meta(self, nodo): return nodo == self.meta
    def vecinos(self, nodo):
        r, c = nodo
        result = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < self.filas and 0 <= nc < self.cols and self.grid[nr][nc] != '#':
                result.append((nr, nc))
        return result

print("Motor de búsqueda listo.")


---
## Sección 1: Generador de Laberintos con Recursive Backtracking

El algoritmo de recursive backtracking es una forma elegante de generar laberintos perfectos.
Su estructura es idéntica a DFS: mantiene una pila de celdas a visitar y retrocede
cuando no encuentra celdas no visitadas.

### Representación

Usamos una cuadrícula de tamaño $(2f+1) \times (2c+1)$:
- Las celdas de la cuadrícula lógica ocupan posiciones $(2r+1, 2c+1)$
- Los muros entre celdas adyacentes ocupan las posiciones intermedias

Al tallar un pasaje entre la celda $(r_1, c_1)$ y $(r_2, c_2)$, abrimos el muro
en la posición intermedia $(r_1+r_2+1, c_1+c_2+1)$ (promedio × 2, simplificado).

```
Cuadrícula 3×3 lógica → 7×7 física:
#  #  #  #  #  #  #
#  .  ?  .  ?  .  #    ? = muro potencial entre celdas
#  ?  #  ?  #  ?  #
#  .  ?  .  ?  .  #
#  ?  #  ?  #  ?  #
#  .  ?  .  ?  .  #
#  #  #  #  #  #  #
```


In [ ]:
def generar_laberinto(filas, cols, seed=42):
    """
    Genera un laberinto perfecto usando recursive backtracking.
    
    El laberinto tiene dimensiones (2*filas+1) x (2*cols+1) para incluir las paredes.
    '#' = pared, '.' = pasaje libre.
    """
    import random
    rng = random.Random(seed)
    h, w = 2 * filas + 1, 2 * cols + 1
    grid = [['#'] * w for _ in range(h)]

    def carve(r, c):
        """DFS recursivo: talla pasajes desde la celda lógica (r, c)."""
        grid[2*r+1][2*c+1] = '.'
        dirs = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        rng.shuffle(dirs)
        for dr, dc in dirs:
            nr, nc = r + dr, c + dc
            if 0 <= nr < filas and 0 <= nc < cols and grid[2*nr+1][2*nc+1] == '#':
                # Abrir el muro entre (r,c) y (nr,nc)
                grid[2*r+1+dr][2*c+1+dc] = '.'
                carve(nr, nc)

    carve(0, 0)
    return grid


# Generar laberinto 15×15 (cuadrícula física 31×31)
import sys
sys.setrecursionlimit(10000)  # necesario para laberintos grandes

lab = generar_laberinto(15, 15, seed=42)
h_lab, w_lab = len(lab), len(lab[0])
print(f"Laberinto generado: {h_lab} × {w_lab} celdas físicas")
print(f"Celdas lógicas: 15 × 15 = 225")
n_libre = sum(lab[r][c] == '.' for r in range(h_lab) for c in range(w_lab))
n_pared = h_lab * w_lab - n_libre
print(f"Celdas libres: {n_libre}, paredes: {n_pared}")
print()
print("Esquina superior-izquierda del laberinto (6×6):")
for fila in lab[:6]:
    print(''.join(fila[:6]))


In [ ]:
def dibujar_laberinto(grid, camino=None, titulo='Laberinto', ax=None,
                       color_camino=None, inicio=None, meta=None):
    """Visualiza un laberinto (grid de '#' y '.') con matplotlib."""
    if color_camino is None:
        color_camino = [0.53, 0.81, 0.98]  # azul claro
    filas, cols = len(grid), len(grid[0])
    img = np.ones((filas, cols, 3))

    # Colorear paredes
    for r in range(filas):
        for c in range(cols):
            if grid[r][c] == '#':
                img[r, c] = [0.15, 0.18, 0.22]

    # Camino
    if camino:
        for (r, c) in camino:
            img[r, c] = color_camino

    # Inicio y meta
    if inicio: img[inicio[0], inicio[1]] = [0.98, 0.57, 0.09]
    if meta:   img[meta[0],   meta[1]]   = [0.13, 0.77, 0.35]

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, interpolation='nearest', aspect='equal')
    ax.set_title(titulo, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    return ax


# Visualizar el laberinto vacío
inicio_lab = (1, 1)  # celda lógica (0,0) en coordenadas físicas = (1,1)
meta_lab   = (2*15-1, 2*15-1)  # celda lógica (14,14) = (29,29)

fig, ax = plt.subplots(figsize=(8, 8))
dibujar_laberinto(lab, titulo='Laberinto perfecto 15×15 (recursive backtracking)',
                  ax=ax, inicio=inicio_lab, meta=meta_lab)
plt.tight_layout()
plt.show()
print(f"Inicio: {inicio_lab} (S=naranja)  |  Meta: {meta_lab} (G=verde)")


---
## Sección 2: Resolución con BFS

BFS explora el laberinto en "capas" concéntricas alrededor del inicio.
Como el laberinto perfecto es un árbol, el camino encontrado es el único camino posible
— y por tanto también el más corto.

En un laberinto perfecto de $n \times n$ celdas, el camino de BFS tiene longitud
proporcional a $n$ en promedio (recorre aproximadamente la mitad del árbol).


In [ ]:
prob_bfs = ProblemaCuadricula(lab, inicio_lab, meta_lab)
camino_bfs, exp_bfs, _ = bfs(prob_bfs)

print(f"BFS — Laberinto 15×15")
print(f"  Longitud del camino:  {len(camino_bfs)} pasos")
print(f"  Nodos expandidos:     {exp_bfs}")

fig, ax = plt.subplots(figsize=(8, 8))
dibujar_laberinto(lab, camino=camino_bfs,
                  titulo=f'BFS — camino de {len(camino_bfs)} pasos ({exp_bfs} nodos expandidos)',
                  ax=ax, inicio=inicio_lab, meta=meta_lab,
                  color_camino=[0.53, 0.81, 0.98])
plt.tight_layout()
plt.show()


---
## Sección 3: Resolución con DFS

DFS se sumerge profundamente en el laberinto siguiendo un brazo hasta el fondo antes de
explorar alternativas. Visualmente, su camino tiene un aspecto más "lineal" y puede ser
muy diferente al de BFS.

Una observación importante: en un laberinto **perfecto** (árbol), **ambos algoritmos
encontrarán el mismo camino** porque hay exactamente uno. En laberintos con ciclos,
los caminos pueden diferir.

En la práctica, la diferencia real entre BFS y DFS en laberintos está en el **orden
de expansión** y en el **tamaño de la frontera**, no en la calidad del camino final
(cuando el laberinto es perfecto).


In [ ]:
prob_dfs = ProblemaCuadricula(lab, inicio_lab, meta_lab)
camino_dfs, exp_dfs, _ = dfs(prob_dfs)

print(f"DFS — Laberinto 15×15")
print(f"  Longitud del camino:  {len(camino_dfs)} pasos")
print(f"  Nodos expandidos:     {exp_dfs}")
print()
if len(camino_bfs) == len(camino_dfs):
    print("✓ Laberinto perfecto: BFS y DFS encuentran el mismo camino único.")
else:
    print(f"Diferencia en longitud: {abs(len(camino_dfs) - len(camino_bfs))} pasos")


In [ ]:
# Visualizar ambos caminos lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

dibujar_laberinto(lab, camino=camino_bfs,
                  titulo=f'BFS — {len(camino_bfs)} pasos, {exp_bfs} expandidos',
                  ax=axes[0], inicio=inicio_lab, meta=meta_lab,
                  color_camino=[0.53, 0.81, 0.98])

dibujar_laberinto(lab, camino=camino_dfs,
                  titulo=f'DFS — {len(camino_dfs)} pasos, {exp_dfs} expandidos',
                  ax=axes[1], inicio=inicio_lab, meta=meta_lab,
                  color_camino=[1.0, 0.60, 0.40])

plt.suptitle('Comparación BFS vs DFS — Laberinto perfecto 15×15', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


---
## Sección 4: Superposición de Caminos

Veamos qué celdas usa cada algoritmo y cuáles comparten.
Usamos un código de colores:
- **Azul**: celdas solo en el camino BFS
- **Rojo**: celdas solo en el camino DFS
- **Morado**: celdas compartidas por ambos caminos

En un laberinto perfecto, los caminos son idénticos, así que esperamos todo en morado.
En laberintos con ciclos podría haber diferencias.


In [ ]:
set_bfs = set(camino_bfs)
set_dfs = set(camino_dfs)
solo_bfs  = set_bfs - set_dfs
solo_dfs  = set_dfs - set_bfs
comunes   = set_bfs & set_dfs

print(f"Celdas exclusivas de BFS:     {len(solo_bfs)}")
print(f"Celdas exclusivas de DFS:     {len(solo_dfs)}")
print(f"Celdas compartidas:           {len(comunes)}")

# Construir imagen con overlay
h_lab2, w_lab2 = len(lab), len(lab[0])
img_overlay = np.ones((h_lab2, w_lab2, 3))
for r in range(h_lab2):
    for c in range(w_lab2):
        if lab[r][c] == '#':
            img_overlay[r, c] = [0.15, 0.18, 0.22]

for (r, c) in solo_bfs:  img_overlay[r, c] = [0.20, 0.50, 0.95]  # azul
for (r, c) in solo_dfs:  img_overlay[r, c] = [0.95, 0.25, 0.25]  # rojo
for (r, c) in comunes:   img_overlay[r, c] = [0.55, 0.20, 0.75]  # morado

img_overlay[inicio_lab[0], inicio_lab[1]] = [0.98, 0.57, 0.09]
img_overlay[meta_lab[0],   meta_lab[1]]   = [0.13, 0.77, 0.35]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(img_overlay, interpolation='nearest', aspect='equal')
ax.set_title('Superposición BFS (azul) vs DFS (rojo) — compartido (morado)', fontsize=12)
ax.set_xticks([]); ax.set_yticks([])

parches = [
    mpatches.Patch(color=[0.20, 0.50, 0.95], label=f'Solo BFS ({len(solo_bfs)})'),
    mpatches.Patch(color=[0.95, 0.25, 0.25], label=f'Solo DFS ({len(solo_dfs)})'),
    mpatches.Patch(color=[0.55, 0.20, 0.75], label=f'Compartido ({len(comunes)})'),
    mpatches.Patch(color=[0.98, 0.57, 0.09], label='Inicio'),
    mpatches.Patch(color=[0.13, 0.77, 0.35], label='Meta'),
]
ax.legend(handles=parches, loc='upper right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()


---
## Sección 5: Escalabilidad — BFS vs DFS por Tamaño de Laberinto

¿Cómo crece el número de nodos expandidos cuando aumentamos el tamaño del laberinto?

Para un laberinto perfecto de $n \times n$ celdas lógicas:
- Hay $(2n+1)^2$ celdas físicas
- De esas, exactamente $n^2$ son celdas lógicas (pasajes de celda)
- El árbol tiene $n^2$ nodos

Ambos algoritmos deben explorar el árbol entero en el peor caso, por lo que esperamos
que los nodos expandidos crezcan como $O(n^2)$.

Sin embargo, BFS y DFS pueden diferir mucho en la **fracción** del árbol explorada
antes de encontrar la solución — esto depende de la estructura del laberinto.


In [ ]:
tamanos = [5, 10, 15, 20, 25]
resultados = []

for n in tamanos:
    lab_n = generar_laberinto(n, n, seed=42)
    ini_n = (1, 1)
    met_n = (2*n-1, 2*n-1)
    
    prob_n = ProblemaCuadricula(lab_n, ini_n, met_n)
    
    t0 = time.perf_counter()
    cam_bfs, exp_bfs_n, _ = bfs(prob_n)
    t_bfs = time.perf_counter() - t0
    
    t0 = time.perf_counter()
    cam_dfs, exp_dfs_n, _ = dfs(prob_n)
    t_dfs = time.perf_counter() - t0
    
    resultados.append({
        'n': n,
        'celdas': n*n,
        'exp_bfs': exp_bfs_n,
        'exp_dfs': exp_dfs_n,
        't_bfs':   t_bfs * 1000,
        't_dfs':   t_dfs * 1000,
        'len_bfs': len(cam_bfs),
        'len_dfs': len(cam_dfs),
    })
    print(f"n={n:2d}: BFS {exp_bfs_n:5d} expandidos ({t_bfs*1000:.1f}ms)"
          f"  |  DFS {exp_dfs_n:5d} expandidos ({t_dfs*1000:.1f}ms)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ns      = [r['n'] for r in resultados]
celdas  = [r['celdas'] for r in resultados]
exp_bfs_list = [r['exp_bfs'] for r in resultados]
exp_dfs_list = [r['exp_dfs'] for r in resultados]

# Nodos expandidos vs tamaño
axes[0].plot(ns, exp_bfs_list, 'o-', color=COLORS["primary"], linewidth=2.5,
             markersize=8, label='BFS')
axes[0].plot(ns, exp_dfs_list, 's--', color=COLORS["danger"], linewidth=2.5,
             markersize=8, label='DFS')
axes[0].plot(ns, celdas, 'k:', linewidth=1.5, alpha=0.5, label='$n^2$ (total celdas)')
axes[0].set_xlabel('Tamaño del laberinto ($n$)', fontsize=12)
axes[0].set_ylabel('Nodos expandidos', fontsize=12)
axes[0].set_title('Nodos expandidos vs tamaño', fontsize=12)
axes[0].legend(fontsize=11)

# Crecimiento cuadrático
axes[1].plot(celdas, exp_bfs_list, 'o-', color=COLORS["primary"], linewidth=2.5,
             markersize=8, label='BFS')
axes[1].plot(celdas, exp_dfs_list, 's--', color=COLORS["danger"], linewidth=2.5,
             markersize=8, label='DFS')
# Línea y=x de referencia
axes[1].plot(celdas, celdas, 'k:', linewidth=1.5, alpha=0.5, label='$y = x$ (lineal)')
axes[1].set_xlabel('Número de celdas $n^2$', fontsize=12)
axes[1].set_ylabel('Nodos expandidos', fontsize=12)
axes[1].set_title('Escalabilidad vs total de celdas', fontsize=12)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\nObservación: los nodos expandidos crecen linealmente con n² — O(n²) total,")
print("lo que confirma que ambos algoritmos recorren el árbol del laberinto completo.")


---
## 🔧 Ejercicios

### Ejercicio 1: Laberintos con ciclos

Los laberintos perfectos son árboles. Modifica el generador para crear un laberinto
**con ciclos** eliminando aleatoriamente el $p\%$ de las paredes interiores.

```python
def agregar_ciclos(grid, porcentaje=0.1, seed=0):
    """Elimina `porcentaje` de las paredes interiores para crear ciclos."""
    import random
    rng = random.Random(seed)
    filas, cols = len(grid), len(grid[0])
    grid_copia = [fila[:] for fila in grid]
    # Tu código aquí: identificar paredes interiores y eliminar algunas
    return grid_copia
```

Prueba con $p \in \{0.05, 0.1, 0.2\}$. ¿Cómo cambia la diferencia en longitud
entre el camino de BFS y el de DFS?

### Ejercicio 2: Visualización de la frontera BFS

BFS expande los nodos en orden de distancia al inicio. Modifica `busqueda_generica`
para guardar el **orden de expansión** de cada nodo:

```python
orden_expansion = {}  # nodo → número de paso en que fue expandido
```

Luego visualiza el laberinto coloreando cada celda según su orden de expansión
(usa un colormap tipo `viridis` o `plasma`). El resultado debe verse como "ondas"
expandiéndose desde el inicio — las celdas más cercanas en color frío, las más
lejanas en color cálido.

Compara la visualización de BFS vs DFS: ¿cómo difieren los patrones?

### Ejercicio 3: Generador de laberintos con Prim aleatorio

Existe otra forma de generar laberintos perfectos: el algoritmo de Prim aleatorio.
En lugar de DFS, usamos una cola de prioridad con pesos aleatorios:

1. Agregar la celda inicial a la frontera
2. Mientras haya paredes en la frontera:
   a. Elegir una pared aleatoria de la frontera
   b. Si exactamente uno de sus lados está visitado, abrir la pared y agregar
      la celda nueva a la frontera
   c. Si no, descartar la pared

Implementa `generar_laberinto_prim(filas, cols, seed)` y compara visualmente
los laberintos generados con Prim vs recursive backtracking. ¿Cuál tiene
ramas más cortas? ¿Cuál produce caminos más largos de BFS?
